# Derivatives Matching Engine — PySpark / Databricks Production

**Document Reference:** Derivatives.docx (Business Requirements Document)

## Purpose
Production-grade PySpark implementation of the Derivatives Trade Matching Engine.
Replicates **exactly the same rules and logic** as the Pandas POC but optimised for:
- **4 M Axis × 20 M Finstore** rows
- **100+ columns** per dataset
- Databricks / Spark cluster execution

## Architecture (Best Practices Applied)
| Principle | Implementation |
|---|---|
| Match narrow, enrich wide | Core schemas (~15 cols) for matching; wide tables joined at the end |
| Candidates → Score → Resolve | All 15 BRD rules generate candidate edges; resolved via window ranking |
| No iterative pool removal | Priority + window ranking replaces repeated anti-joins |
| Block aggressively | Counterparty blocking (Strategy 1), amount-bucket blocking (Strategy 2) |
| Delta/Parquet everywhere | CSV only as final export |
| No Python UDFs | All derivations use native Spark SQL functions |

## Layers
- **Layer 1:** BRD Deterministic Rules (15 total: 3 SOPHIS + 10 OTC + 2 ETD)
- **Layer 2:** Greedy / Probabilistic Matching
  - Strategy 1: Amount + Counterparty (1% tolerance)
  - Strategy 2: Amount-only strict (0.1% tolerance)

## Scope Note
SOPHIS and DELTA1 systems are excluded by default.
Their trade IDs do not exist in Finstore — this is a data mapping issue, not a matching algorithm issue.

**Date:** 2026-02-23 (v4 — PySpark Production)

---
## 🔹 Section 1: Spark Session & Configuration

In [ ]:
# ============================================================
# SPARK SESSION & IMPORTS
# ============================================================
# On Databricks the SparkSession is pre-created as `spark`.
# For local testing uncomment the builder block below.

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from functools import reduce
from datetime import datetime
import logging

# --- Uncomment for local / non-Databricks execution ---
# spark = (
#     SparkSession.builder
#     .master("local[*]")
#     .appName("DerivativesMatchingEngine")
#     .config("spark.sql.adaptive.enabled", "true")
#     .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
#     .config("spark.sql.adaptive.skewJoin.enabled", "true")
#     .config("spark.sql.shuffle.partitions", "200")
#     .config("spark.driver.memory", "8g")
#     .getOrCreate()
# )

# Ensure AQE is enabled (Databricks default, but explicit is safer)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

print(f"Spark version: {spark.version}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Skew join: {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")

In [ ]:
# ============================================================
# SCOPE CONFIGURATION
# ============================================================

# Set to True to exclude SOPHIS and DELTA1 systems from scope
# These systems' trade IDs don't exist in Finstore (data mapping issue)
EXCLUDE_SOPHIS_DELTA1 = True

# Out-of-scope systems (when EXCLUDE_SOPHIS_DELTA1 = True)
OUT_OF_SCOPE_SYSTEMS = [
    "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO", "SOPHISFX-LONDON",
    "DELTA1-LONDON", "DELTA1-NEWYORK"
]

# Greedy matching parameters
GREEDY_TOLERANCE_PCT = 0.01     # 1% for Strategy 1 (Amount + Counterparty)
STRICT_TOLERANCE_PCT = 0.001    # 0.1% for Strategy 2 (Amount only)
BUCKET_SIZE = 1000              # Amount bucket size for Strategy 2 blocking

# ============================================================
# LINEAGE & AUDITABILITY  (Best Practice §6)
# ============================================================
# Every match row will carry these for full traceability.
import uuid

RUN_ID = str(uuid.uuid4())                           # unique per execution
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")  # human-readable batch
RUN_TIMESTAMP = datetime.now().isoformat()
RULE_VERSION = "v4.0-pyspark"                         # bump on logic changes

# ============================================================
# FILE PATHS — Adjust for Databricks / ADLS / DBFS
# ============================================================

# For Databricks: use dbfs:/ or abfss:// paths
# For local: use file:///path/to/files
INPUT_DIR = "/mnt/data/derivatives"           # <-- adjust to your mount / path
INPUT_FILE_AXIS = f"{INPUT_DIR}/axis_sample_poc.csv"
INPUT_FILE_FINSTORE = f"{INPUT_DIR}/finstore_sample_poc.csv"
SDS_MAPPING_FILE = f"{INPUT_DIR}/sds_book_mapping.csv"

OUTPUT_DIR = f"{INPUT_DIR}/matching_results"

print("SCOPE CONFIGURATION:")
print(f"  EXCLUDE_SOPHIS_DELTA1: {EXCLUDE_SOPHIS_DELTA1}")
if EXCLUDE_SOPHIS_DELTA1:
    print(f"  Excluded systems: {', '.join(OUT_OF_SCOPE_SYSTEMS)}")
print(f"\nGREEDY MATCHING PARAMETERS:")
print(f"  Strategy 1 tolerance: {GREEDY_TOLERANCE_PCT * 100}%")
print(f"  Strategy 2 tolerance: {STRICT_TOLERANCE_PCT * 100}%")
print(f"  Bucket size: {BUCKET_SIZE}")
print(f"\nLINEAGE:")
print(f"  run_id:       {RUN_ID}")
print(f"  batch_id:     {BATCH_ID}")
print(f"  rule_version: {RULE_VERSION}")
print(f"\nPATHS:")
print(f"  Input:  {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")

---
## 🔹 Section 2: BRD Constants & System Classifications

In [ ]:
# ============================================================
# BRD SECTION: System Classification Lists
# ============================================================

# OTC Systems (from BRD table)
OTC_SYSTEMS = [
    "ALD-SF", "BARXFX-PD-ASIAPAC", "BARXFX-PD-LONDON", "BARXFETS",
    "BARXFX-TS-CASH-ASIAPAC", "BARXFX-TS-CASH-LONDON", "DELTA1-LONDON",
    "DELTA1-NEWYORK", "IFLOW-AMERICAS", "IFLOW-ASIAPACIFIC", "IFLOW-EUROPE",
    "FISSFX-NEWYORK", "FISS-LONDON", "FISS-TOKYO", "GCD-NEWYORK",
    "IMPACT-NEWYORK", "OPENLINK-LONDON", "OPTISRD-MEXICO", "SABS-NEWYORK",
    "SOPHISFX-LONDON", "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO",
    "SUMMIT-LONDON", "SYNFINY-CFD-NONCASH", "SYETR-NEWYORK-MBS"
]

# ETD Systems (from BRD table)
ETD_SYSTEMS = [
    "BPS222-OPT-NEWYORK", "BPS224-OPT-NEWYORK", "ODH-DOLPHIN-INDIA",
    "ODH-CMI-CPM-LONDON", "ODH-GMI-HONGKONG", "ODH-GMI-LONDON",
    "ODH-GMI-LONDON-BDT", "ODH-GMI-NEWYORK", "ODH-GMI-SINGAPORE",
    "ODH-ISTAR-TOKYO", "OPTISFUT-MEXICO"
]

# SOPHIS Systems
SOPHIS_SYSTEMS = [
    "SOPHISFX-LONDON", "SOPHIS-LONDON", "SOPHIS-TOKYO", "SOPHIS-NEWYORK"
]

# DELTA1 Systems
DELTA1_SYSTEMS = ["DELTA1-LONDON", "DELTA1-NEWYORK"]

# Amount column names
AXIS_AMOUNT_COL = "SACCRMTM"
FIN_AMOUNT_COL = "gbpequivalentamount"

print("BRD CONSTANTS LOADED")
print(f"  OTC Systems:    {len(OTC_SYSTEMS)}")
print(f"  ETD Systems:    {len(ETD_SYSTEMS)}")
print(f"  SOPHIS Systems: {len(SOPHIS_SYSTEMS)}")
print(f"  DELTA1 Systems: {len(DELTA1_SYSTEMS)}")

---
## 🔹 Section 3: Matching Rule Definitions (All 15 BRD Rules)

Rules are defined as lightweight dictionaries instead of dataclasses.
Each rule specifies:
- `priority`: global waterfall order (1 = highest priority)
- `category`: SOPHIS / OTC / ETD
- `brd_priority`: priority within category
- `axis_keys` / `fin_keys`: column(s) to join on
- `filter_sophis_only`: whether to restrict Axis to SOPHIS systems
- `requires_derived_masterbook`: skipped if DerivedMasterbookId is empty

In [ ]:
# ============================================================
# ALL 15 BRD MATCHING RULES
# ============================================================
# Exact same rules as the Pandas notebook, expressed as dicts
# for easy iteration in Spark.

MATCHING_RULES = [
    # ---- SOPHIS-specific rules (Priority 1-3) ----
    dict(
        priority=1, category="SOPHIS", brd_priority=1,
        description="DerivedSophisId <> fissnumber",
        axis_keys=["DerivedSophisId"],
        fin_keys=["fissnumber"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=2, category="SOPHIS", brd_priority=2,
        description="DerivedSophisId + BookId <> fissnumber + tradingsystembook",
        axis_keys=["DerivedSophisId", "BookId"],
        fin_keys=["fissnumber", "tradingsystembook"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=3, category="SOPHIS", brd_priority=3,
        description="DerivedSophisId <> tradeid",
        axis_keys=["DerivedSophisId"],
        fin_keys=["tradeid"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),

    # ---- OTC rules (Priority 4-13) ----
    dict(
        priority=4, category="OTC", brd_priority=1,
        description="SourceSystemTradeId <> tradeid",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=5, category="OTC", brd_priority=2,
        description="SourceSystemTradeId + DerivedMasterbookId <> tradeid + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["tradeid", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=6, category="OTC", brd_priority=3,
        description="SourceSystemTradeId <> alternatetradeid1",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["alternatetradeid1"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=7, category="OTC", brd_priority=4,
        description="SourceSystemTradeId + DerivedMasterbookId <> alternatetradeid1 + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["alternatetradeid1", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=8, category="OTC", brd_priority=5,
        description="DerivedSophisId <> fissnumber",
        axis_keys=["DerivedSophisId"],
        fin_keys=["fissnumber"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=9, category="OTC", brd_priority=6,
        description="DerivedSophisId + BookId <> fissnumber + tradingsystembook",
        axis_keys=["DerivedSophisId", "BookId"],
        fin_keys=["fissnumber", "tradingsystembook"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=10, category="OTC", brd_priority=7,
        description="DerivedSophisId <> tradeid",
        axis_keys=["DerivedSophisId"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=11, category="OTC", brd_priority=8,
        description="DerivedDelta1Id <> tradeid",
        axis_keys=["DerivedDelta1Id"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=12, category="OTC", brd_priority=9,
        description="SourceSystemTradeId <> alternatetradeid2",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["alternatetradeid2"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=13, category="OTC", brd_priority=10,
        description="SourceSystemTradeId + DerivedMasterbookId <> alternatetradeid2 + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["alternatetradeid2", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),

    # ---- ETD rules (Priority 14-15) ----
    dict(
        priority=14, category="ETD", brd_priority=1,
        description="SourceSystemInstrumentId + DerivedMasterbookId <> instrumentid + masterbookid",
        axis_keys=["SourceSystemInstrumentId", "DerivedMasterbookId"],
        fin_keys=["instrumentid", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=15, category="ETD", brd_priority=2,
        description="SourceSystemInstrumentId <> instrumentid",
        axis_keys=["SourceSystemInstrumentId"],
        fin_keys=["instrumentid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
]

sophis_count = sum(1 for r in MATCHING_RULES if r["category"] == "SOPHIS")
otc_count = sum(1 for r in MATCHING_RULES if r["category"] == "OTC")
etd_count = sum(1 for r in MATCHING_RULES if r["category"] == "ETD")

print(f"SOPHIS rules: {sophis_count}")
print(f"OTC rules:    {otc_count}")
print(f"ETD rules:    {etd_count}")
print(f"TOTAL:        {len(MATCHING_RULES)} rules")

---
## 🔹 Section 4: Load Data (Bronze Layer)

In [ ]:
# ============================================================
# BRONZE LAYER — Raw Ingestion
# ============================================================
# For production: read from Delta tables instead of CSV.
# Example: spark.read.format("delta").load("dbfs:/mnt/...")

logger.info("Loading Axis data...")
df_axis_full = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(INPUT_FILE_AXIS)
)
# Bronze metadata (Best Practice §2A — append-only lineage)
df_axis_full = (
    df_axis_full
    .withColumn("_ingest_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("_source_file", F.lit(INPUT_FILE_AXIS))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)
ORIGINAL_AXIS_COUNT_FULL = df_axis_full.count()
logger.info(f"Axis loaded: {ORIGINAL_AXIS_COUNT_FULL:,} records")

logger.info("Loading Finstore data...")
df_finstore_full = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(INPUT_FILE_FINSTORE)
)
# Bronze metadata
df_finstore_full = (
    df_finstore_full
    .withColumn("_ingest_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("_source_file", F.lit(INPUT_FILE_FINSTORE))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)
ORIGINAL_FINSTORE_COUNT = df_finstore_full.count()
logger.info(f"Finstore loaded: {ORIGINAL_FINSTORE_COUNT:,} records")

# Load SDS mapping if available (small table — good broadcast candidate)
try:
    df_sds_mapping = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(SDS_MAPPING_FILE)
    )
    sds_available = True
    logger.info(f"SDS mapping loaded: {df_sds_mapping.count():,} records")
except Exception:
    df_sds_mapping = None
    sds_available = False
    logger.info("SDS mapping not available — DerivedMasterbookId will be empty")

print(f"\n{'='*80}")
print("DATA LOADED (BRONZE)")
print(f"{'='*80}")
print(f"Axis (full):   {ORIGINAL_AXIS_COUNT_FULL:,} records, {len(df_axis_full.columns)} columns")
print(f"Finstore:      {ORIGINAL_FINSTORE_COUNT:,} records, {len(df_finstore_full.columns)} columns")
print(f"SDS Mapping:   {'Available' if sds_available else 'NOT AVAILABLE'}")

---
## 🔹 Section 5: Scope Exclusion (SOPHIS / DELTA1)

In [ ]:
# ============================================================
# SCOPE EXCLUSION
# ============================================================

print(f"\n{'='*80}")
print("SCOPE EXCLUSION")
print(f"{'='*80}")

if EXCLUDE_SOPHIS_DELTA1:
    df_axis_excluded = df_axis_full.filter(
        F.col("SourceSystemName").isin(OUT_OF_SCOPE_SYSTEMS)
    )
    df_axis = df_axis_full.filter(
        ~F.col("SourceSystemName").isin(OUT_OF_SCOPE_SYSTEMS)
    )

    excluded_count = df_axis_excluded.count()
    in_scope_count = df_axis.count()

    print("\n*** SOPHIS/DELTA1 SCOPE EXCLUSION ACTIVE ***")
    print("\nExcluded systems breakdown:")
    df_axis_excluded.groupBy("SourceSystemName").count().orderBy(F.desc("count")).show(20, truncate=False)

    print(f"TOTAL EXCLUDED: {excluded_count:,}")
    print(f"\nScope Summary:")
    print(f"  Original Axis: {ORIGINAL_AXIS_COUNT_FULL:,}")
    print(f"  Excluded:      {excluded_count:,} ({excluded_count/ORIGINAL_AXIS_COUNT_FULL*100:.1f}%)")
    print(f"  In-Scope:      {in_scope_count:,} ({in_scope_count/ORIGINAL_AXIS_COUNT_FULL*100:.1f}%)")
else:
    df_axis = df_axis_full
    df_axis_excluded = spark.createDataFrame([], df_axis_full.schema)
    in_scope_count = ORIGINAL_AXIS_COUNT_FULL
    excluded_count = 0
    print(f"All systems in scope: {in_scope_count:,}")

ORIGINAL_AXIS_COUNT = in_scope_count
print(f"\nWorking Axis count (in-scope): {ORIGINAL_AXIS_COUNT:,}")

---
## 🔹 Section 6: Pre-Reconciliation Derivations (Silver Layer)

All derivations use **native Spark SQL functions** — no Python UDFs.
- `DerivedSophisId`: 3rd part of hyphen-split `SourceSystemTradeId` (SOPHIS systems)
- `DerivedDelta1Id`: 3rd part of hyphen-split `SourceSystemTradeId` (DELTA1 systems)
- `ReconSubProduct`: ETD / OTC / OTC-Default classification
- `DerivedMasterbookId`: from SDS mapping (if available)

In [ ]:
# ============================================================
# SILVER LAYER — Derivations & Key Preparation
# ============================================================
# All done with native Spark SQL — no Python UDFs.

# --- Stable surrogate IDs (monotonically_increasing_id is unique per run) ---
df_axis = df_axis.withColumn("axis_id", F.monotonically_increasing_id())
df_finstore = df_finstore_full.withColumn("fin_id", F.monotonically_increasing_id())

# --- ReconSubProduct classification ---
df_axis = df_axis.withColumn(
    "ReconSubProduct",
    F.when(F.col("SourceSystemName").isin(ETD_SYSTEMS), F.lit("ETD"))
     .when(F.col("SourceSystemName").isin(OTC_SYSTEMS), F.lit("OTC"))
     .otherwise(F.lit("OTC-Default"))
)

# --- DerivedSophisId: extract 3rd part of hyphen-separated SourceSystemTradeId ---
# Native Spark: split() + element_at() — no UDF needed
df_axis = df_axis.withColumn(
    "DerivedSophisId",
    F.when(
        F.col("SourceSystemName").isin(SOPHIS_SYSTEMS) &
        F.col("SourceSystemTradeId").isNotNull() &
        (F.size(F.split(F.col("SourceSystemTradeId"), "-")) >= 3),
        F.element_at(F.split(F.col("SourceSystemTradeId"), "-"), 3)
    ).otherwise(F.lit(""))
)

# --- DerivedDelta1Id: same extraction for DELTA1 systems ---
df_axis = df_axis.withColumn(
    "DerivedDelta1Id",
    F.when(
        F.col("SourceSystemName").isin(DELTA1_SYSTEMS) &
        F.col("SourceSystemTradeId").isNotNull() &
        (F.size(F.split(F.col("SourceSystemTradeId"), "-")) >= 3),
        F.element_at(F.split(F.col("SourceSystemTradeId"), "-"), 3)
    ).otherwise(F.lit(""))
)

# --- DerivedMasterbookId placeholder (requires SDS mapping) ---
# TODO: When SDS is available, do a broadcast join here:
# df_axis = df_axis.join(broadcast(df_sds_mapping), on=<key>, how="left")
df_axis = df_axis.withColumn("DerivedMasterbookId", F.lit(""))

# --- Ensure amount columns are numeric ---
df_axis = df_axis.withColumn(AXIS_AMOUNT_COL, F.col(AXIS_AMOUNT_COL).cast("double"))
df_finstore = df_finstore.withColumn(FIN_AMOUNT_COL, F.col(FIN_AMOUNT_COL).cast("double"))

# --- Ensure string join columns are trimmed ---
string_cols_axis = [
    "SourceSystemTradeId", "BookId", "DerivedSophisId", "DerivedDelta1Id",
    "DerivedMasterbookId", "SourceSystemInstrumentId", "CounterpartyId"
]
for c in string_cols_axis:
    if c in df_axis.columns:
        df_axis = df_axis.withColumn(c, F.trim(F.col(c).cast("string")))

string_cols_fin = [
    "tradeid", "alternatetradeid1", "alternatetradeid2", "masterbookid",
    "tradingsystembook", "fissnumber", "instrumentid", "counterpartyid"
]
for c in string_cols_fin:
    if c in df_finstore.columns:
        df_finstore = df_finstore.withColumn(c, F.trim(F.col(c).cast("string")))

print("PRE-RECONCILIATION DERIVATIONS COMPLETE")
print("\nReconSubProduct Distribution:")
df_axis.groupBy("ReconSubProduct").count().orderBy(F.desc("count")).show()
print(f"DerivedSophisId populated: {df_axis.filter(F.col('DerivedSophisId') != '').count():,}")
print(f"DerivedDelta1Id populated: {df_axis.filter(F.col('DerivedDelta1Id') != '').count():,}")

---
## 🔹 Section 7: Core / Wide Split (Match Narrow, Enrich Wide)

**Critical performance optimisation:** only carry the ~15 columns needed for matching.
The remaining 100+ columns stay in `_wide` tables and are joined back at the end.

In [ ]:
# ============================================================
# CORE / WIDE SPLIT — minimise shuffle payload
# ============================================================

# Axis core: only columns needed for matching
AXIS_CORE_COLS = [
    "axis_id", "SourceSystemName", "SourceSystemTradeId", "BookId",
    "CounterpartyId", AXIS_AMOUNT_COL, "SourceSystemInstrumentId",
    "DerivedSophisId", "DerivedDelta1Id", "DerivedMasterbookId",
    "ReconSubProduct",
]
# Filter to columns that actually exist
AXIS_CORE_COLS = [c for c in AXIS_CORE_COLS if c in df_axis.columns]

axis_core = df_axis.select(AXIS_CORE_COLS)
axis_wide = df_axis  # keep full DataFrame; we join by axis_id at the end

# Finstore core: only columns needed for matching
FIN_CORE_COLS = [
    "fin_id", "tradeid", "alternatetradeid1", "alternatetradeid2",
    "masterbookid", "tradingsystembook", "fissnumber", "instrumentid",
    "counterpartyid", FIN_AMOUNT_COL,
]
FIN_CORE_COLS = [c for c in FIN_CORE_COLS if c in df_finstore.columns]

fin_core = df_finstore.select(FIN_CORE_COLS)
fin_wide = df_finstore  # keep full DataFrame; we join by fin_id at the end

# Cache core tables — they are reused across all rules
axis_core = axis_core.cache()
fin_core = fin_core.cache()

print(f"Axis core columns:    {len(AXIS_CORE_COLS)} → {axis_core.count():,} rows")
print(f"Finstore core columns:{len(FIN_CORE_COLS)} → {fin_core.count():,} rows")
print(f"\nAxis wide columns:    {len(df_axis.columns)}")
print(f"Finstore wide columns:{len(df_finstore.columns)}")

---
## ✅ LAYER 1: BRD DETERMINISTIC MATCHING (15 Rules)

### Architecture: Candidates → Score → Resolve

Instead of executing rules one-by-one and mutating pools (13+ shuffles),
we:
1. **Generate candidate edges** for each rule (one join per rule, narrow schema)
2. **Union all candidates** into a single DataFrame
3. **Resolve 1-to-1** via window ranking (priority first, amount_diff tiebreaker)

This replicates the waterfall "remove matched from pools" behaviour
using a **single ranking pass**, avoiding repeated anti-joins.

---
### 🔹 Section 8: Candidate Generation Function

In [ ]:
# ============================================================
# CANDIDATE GENERATION — one function for all BRD rules
# ============================================================

def build_candidates_for_rule(
    axis_df: DataFrame,
    fin_df: DataFrame,
    rule: dict,
) -> DataFrame:
    """
    Generate candidate match edges for a single BRD rule.
    
    Returns a DataFrame with:
        axis_id, fin_id, priority, category, brd_priority,
        description, amount_diff
    
    Approach:
    - Project only the join keys + IDs + amount (minimise shuffle)
    - Apply system filters if needed (SOPHIS-only rules)
    - Skip rules requiring DerivedMasterbookId if not populated
    - Filter out invalid keys (empty, 'nan', containing '$')
    - Rename axis/fin keys to common names → equi-join
    - Compute amount_diff as tiebreaker
    """
    priority = rule["priority"]
    axis_keys = rule["axis_keys"]
    fin_keys = rule["fin_keys"]

    # --- System filter ---
    a = axis_df
    if rule["filter_sophis_only"]:
        a = a.filter(F.col("SourceSystemName").isin(SOPHIS_SYSTEMS))

    # --- Skip if DerivedMasterbookId required but empty ---
    if rule["requires_derived_masterbook"]:
        non_empty_count = a.filter(F.col("DerivedMasterbookId") != "").limit(1).count()
        if non_empty_count == 0:
            # Return empty DataFrame with correct schema (must match full candidate schema)
            return spark.createDataFrame([], T.StructType([
                T.StructField("axis_id", T.LongType()),
                T.StructField("fin_id", T.LongType()),
                T.StructField("priority", T.IntegerType()),
                T.StructField("category", T.StringType()),
                T.StructField("brd_priority", T.IntegerType()),
                T.StructField("description", T.StringType()),
                T.StructField("amount_diff", T.DoubleType()),
                T.StructField("amount_rel_diff", T.DoubleType()),
                T.StructField("key_strength", T.IntegerType()),
            ]))

    # --- Project to narrow schema ---
    a_cols = ["axis_id", AXIS_AMOUNT_COL] + axis_keys
    a_sel = a.select([F.col(c) for c in a_cols])

    f_cols = ["fin_id", FIN_AMOUNT_COL] + fin_keys
    f_sel = fin_df.select([F.col(c) for c in f_cols])

    # --- Filter out invalid keys (empty, 'nan', containing '$') ---
    for k in axis_keys:
        a_sel = a_sel.filter(
            (F.col(k).isNotNull()) &
            (F.col(k) != "") &
            (F.col(k) != "nan") &
            (~F.col(k).contains("$"))
        )
    for k in fin_keys:
        f_sel = f_sel.filter(
            (F.col(k).isNotNull()) &
            (F.col(k) != "") &
            (F.col(k) != "nan") &
            (~F.col(k).contains("$"))
        )

    # --- Rename keys to common join names (k0, k1, ...) ---
    for i, ak in enumerate(axis_keys):
        a_sel = a_sel.withColumnRenamed(ak, f"_jk{i}")
    for i, fk in enumerate(fin_keys):
        f_sel = f_sel.withColumnRenamed(fk, f"_jk{i}")

    join_cols = [f"_jk{i}" for i in range(len(axis_keys))]

    # --- Join ---
    joined = a_sel.join(f_sel, on=join_cols, how="inner")

    # --- Compute amount_diff (tiebreaker) ---
    joined = joined.withColumn(
        "amount_diff",
        F.abs(F.col(AXIS_AMOUNT_COL) - F.col(FIN_AMOUNT_COL))
    )

    # --- Scoring metrics (Best Practice §2D) ---
    # amount_rel_diff: normalised difference for cross-amount comparability
    joined = joined.withColumn(
        "amount_rel_diff",
        F.col("amount_diff") / F.greatest(F.abs(F.col(AXIS_AMOUNT_COL)), F.lit(1e-9))
    )
    # key_strength: number of join keys matched (higher = more confident)
    key_strength = len(axis_keys)
    joined = joined.withColumn("key_strength", F.lit(key_strength).cast("int"))

    # --- Output standardised candidate schema ---
    return joined.select(
        F.col("axis_id"),
        F.col("fin_id"),
        F.lit(priority).cast("int").alias("priority"),
        F.lit(rule["category"]).alias("category"),
        F.lit(rule["brd_priority"]).cast("int").alias("brd_priority"),
        F.lit(rule["description"]).alias("description"),
        F.col("amount_diff"),
        F.col("amount_rel_diff"),
        F.col("key_strength"),
    )


print("Candidate generation function defined.")

---
### 🔹 Section 9: Generate All BRD Candidates

In [ ]:
# ============================================================
# GENERATE CANDIDATE EDGES FOR ALL 15 BRD RULES
# ============================================================

print(f"\n{'='*80}")
print("LAYER 1: GENERATING BRD CANDIDATES")
print(f"{'='*80}")

candidate_dfs = []

for rule in MATCHING_RULES:
    cand = build_candidates_for_rule(axis_core, fin_core, rule)
    candidate_dfs.append(cand)
    print(f"  Rule {rule['priority']:2d} ({rule['category']:6s} #{rule['brd_priority']}): "
          f"{rule['description']}")

# Union all candidate edges
candidates_layer1 = reduce(DataFrame.unionByName, candidate_dfs)

# Cache — this is reused in resolution
candidates_layer1 = candidates_layer1.cache()

total_candidates = candidates_layer1.count()
print(f"\nTotal candidate edges generated: {total_candidates:,}")
print("\nCandidate counts per rule:")
candidates_layer1.groupBy("priority", "category", "description").count() \
    .orderBy("priority").show(20, truncate=False)

---
### 🔹 Section 10: Resolve 1-to-1 Matches (Waterfall via Window Ranking)

This is the **key performance optimisation**. Instead of 15 iterative
"join → remove matched → repeat" passes, we do **two window passes**:

1. **Best Finstore per Axis:** for each `axis_id`, pick the candidate
   with lowest `priority` (then smallest `amount_diff`, then smallest `fin_id` for stability)
2. **Best Axis per Finstore:** from the reduced set, for each `fin_id`,
   pick the candidate with lowest `priority` (same tiebreakers)

This replicates waterfall removal + prevents double-matching.

In [ ]:
# ============================================================
# 1-to-1 RESOLUTION via Window Ranking
# ============================================================

def resolve_one_to_one(candidates: DataFrame) -> DataFrame:
    """
    Enforce 1-to-1 matching using iterative window ranking.
    
    Pass 1: For each axis_id, pick best fin_id
            (lowest priority → highest key_strength → smallest amount_diff → smallest fin_id)
    Pass 2: For each fin_id, pick best axis_id from Pass 1 result
            (lowest priority → highest key_strength → smallest amount_diff → smallest axis_id)
    Pass 3: (Safety) Re-check axis uniqueness after Pass 2 to handle any
            residual collisions (rare with stable tiebreakers).
    
    Best Practice §2E: iterative resolution with stable tiebreakers.
    """
    # Include key_strength if present (deterministic rules have it)
    has_key_strength = "key_strength" in candidates.columns

    ordering = [
        F.col("priority").asc(),
    ]
    if has_key_strength:
        ordering.append(F.col("key_strength").desc())  # more keys matched = better
    ordering.append(F.col("amount_diff").asc_nulls_last())

    # Pass 1: best fin per axis
    w_axis = Window.partitionBy("axis_id").orderBy(*ordering, F.col("fin_id").asc())
    best_per_axis = (
        candidates
        .withColumn("_rn_axis", F.row_number().over(w_axis))
        .filter(F.col("_rn_axis") == 1)
        .drop("_rn_axis")
    )

    # Pass 2: best axis per fin (prevents fin reuse)
    w_fin = Window.partitionBy("fin_id").orderBy(*ordering, F.col("axis_id").asc())
    pass2 = (
        best_per_axis
        .withColumn("_rn_fin", F.row_number().over(w_fin))
        .filter(F.col("_rn_fin") == 1)
        .drop("_rn_fin")
    )

    # Pass 3: safety — re-enforce axis uniqueness after fin dedup
    # (handles the rare case where pass 2 displaced an axis's original pick)
    w_axis_final = Window.partitionBy("axis_id").orderBy(*ordering, F.col("fin_id").asc())
    resolved = (
        pass2
        .withColumn("_rn_final", F.row_number().over(w_axis_final))
        .filter(F.col("_rn_final") == 1)
        .drop("_rn_final")
    )

    return resolved


print("Resolving 1-to-1 matches...")
brd_matches = resolve_one_to_one(candidates_layer1)

# Add match layer label
brd_matches = brd_matches.withColumn("MatchLayer", F.lit("BRD"))

# Auditability columns (Best Practice §6)
brd_matches = (
    brd_matches
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
)

# Cache for reuse
brd_matches = brd_matches.cache()

brd_match_count = brd_matches.count()
brd_match_rate = brd_match_count / ORIGINAL_AXIS_COUNT * 100

print(f"\n{'='*80}")
print("LAYER 1 (BRD) RESULTS")
print(f"{'='*80}")
print(f"Unique Axis matched: {brd_match_count:,}")
print(f"BRD Match Rate: {brd_match_rate:.2f}%")
print(f"Remaining unmatched: {ORIGINAL_AXIS_COUNT - brd_match_count:,}")

print("\nMatches by rule:")
brd_matches.groupBy("priority", "category", "description").count() \
    .orderBy("priority").show(20, truncate=False)

---
## 🔹 LAYER 2: GREEDY / PROBABILISTIC MATCHING

Applied only to records **unmatched after Layer 1**.

**Key Spark optimisations vs. Pandas:**
- No Python `iterrows()` loops — everything is join-based
- Strategy 1: blocked join on `counterpartyid` + tolerance filter
- Strategy 2: blocked join on amount buckets (±1 bucket) + strict tolerance filter
- 1-to-1 resolution via the same window ranking approach

---
### 🔹 Section 11: Compute Unmatched Pools for Greedy

In [ ]:
# ============================================================
# COMPUTE UNMATCHED POOLS
# ============================================================

# Axis IDs matched in Layer 1
matched_axis_ids = brd_matches.select("axis_id")
matched_fin_ids = brd_matches.select("fin_id")

# Anti-join to get unmatched
axis_unmatched = axis_core.join(matched_axis_ids, on="axis_id", how="left_anti")
fin_unmatched = fin_core.join(matched_fin_ids, on="fin_id", how="left_anti")

# Ensure amounts are clean
axis_unmatched = axis_unmatched.filter(
    F.col(AXIS_AMOUNT_COL).isNotNull() & (F.col(AXIS_AMOUNT_COL) != 0)
)
fin_unmatched = fin_unmatched.filter(
    F.col(FIN_AMOUNT_COL).isNotNull()
)

# Normalise counterparty for join
axis_unmatched = axis_unmatched.withColumn(
    "cpty_str", F.trim(F.col("CounterpartyId").cast("string"))
)
fin_unmatched = fin_unmatched.withColumn(
    "cpty_str", F.trim(F.col("counterpartyid").cast("string"))
)

# Cache
axis_unmatched = axis_unmatched.cache()
fin_unmatched = fin_unmatched.cache()

axis_unmatched_count = axis_unmatched.count()
fin_unmatched_count = fin_unmatched.count()

print(f"\n{'='*80}")
print("LAYER 2: GREEDY / PROBABILISTIC MATCHING")
print(f"{'='*80}")
print(f"Axis remaining for greedy:    {axis_unmatched_count:,}")
print(f"Finstore remaining for greedy:{fin_unmatched_count:,}")

---
### 🔹 Section 12: Greedy Strategy 1 — Amount + Counterparty (1%)

Replaces the Pandas `groupby → iterrows` loop with a **join on counterparty + tolerance filter**.

In [ ]:
# ============================================================
# GREEDY STRATEGY 1: Amount + Counterparty (1% tolerance)
# ============================================================
# Blocked join on counterparty → filter by tolerance → rank best match.

print(f"\n--- Strategy 1: Amount + Counterparty ({GREEDY_TOLERANCE_PCT*100}% tolerance) ---")

# Join on counterparty (blocking key)
greedy1_candidates = (
    axis_unmatched.alias("a")
    .join(
        fin_unmatched.alias("f"),
        on=(F.col("a.cpty_str") == F.col("f.cpty_str")),
        how="inner"
    )
    # Filter: exclude nan/empty counterparties
    .filter(
        (F.col("a.cpty_str").isNotNull()) &
        (F.col("a.cpty_str") != "") &
        (F.col("a.cpty_str") != "nan")
    )
    # Compute difference and tolerance
    .withColumn("amount_diff", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}") - F.col(f"f.{FIN_AMOUNT_COL}")))
    .withColumn("tolerance", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")) * GREEDY_TOLERANCE_PCT)
    # Filter within tolerance
    .filter(F.col("amount_diff") <= F.col("tolerance"))
    # Select candidate edge columns
    .select(
        F.col("a.axis_id").alias("axis_id"),
        F.col("f.fin_id").alias("fin_id"),
        F.lit(16).cast("int").alias("priority"),
        F.lit("GREEDY").alias("category"),
        F.lit(1).cast("int").alias("brd_priority"),
        F.lit(f"Greedy: Amount+Counterparty ({GREEDY_TOLERANCE_PCT*100}%)").alias("description"),
        F.col("amount_diff"),
        (F.col("amount_diff") / F.greatest(F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")), F.lit(1e-9))).alias("amount_rel_diff"),
        F.lit(0).cast("int").alias("key_strength"),  # no key match — fuzzy
    )
)

# Resolve 1-to-1
greedy1_matches = resolve_one_to_one(greedy1_candidates)
greedy1_matches = greedy1_matches.withColumn("MatchLayer", F.lit("GREEDY"))
# Auditability columns (Best Practice §6)
greedy1_matches = (
    greedy1_matches
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
)
greedy1_matches = greedy1_matches.cache()

greedy1_count = greedy1_matches.count()
print(f"Strategy 1 matches: {greedy1_count:,}")

---
### 🔹 Section 13: Greedy Strategy 2 — Amount Only (0.1%)

Uses **amount bucket blocking** to avoid full cross-join.
Each Axis record expands to 3 bucket rows (bucket-1, bucket, bucket+1)
then joins Finstore on bucket.

In [ ]:
# ============================================================
# GREEDY STRATEGY 2: Amount Only (0.1% tolerance) with Bucket Blocking
# ============================================================

print(f"\n--- Strategy 2: Amount only ({STRICT_TOLERANCE_PCT*100}% tolerance) ---")

# Remove records already matched in Strategy 1
greedy1_axis_ids = greedy1_matches.select("axis_id")
greedy1_fin_ids = greedy1_matches.select("fin_id")

axis_remaining_s2 = axis_unmatched.join(greedy1_axis_ids, on="axis_id", how="left_anti")
fin_remaining_s2 = fin_unmatched.join(greedy1_fin_ids, on="fin_id", how="left_anti")

print(f"Axis remaining after Strategy 1: {axis_remaining_s2.count():,}")

# Create amount buckets (native Spark — no UDF)
axis_remaining_s2 = axis_remaining_s2.withColumn(
    "amount_bucket",
    (F.floor(F.col(AXIS_AMOUNT_COL) / BUCKET_SIZE) * BUCKET_SIZE).cast("long")
)

fin_remaining_s2 = fin_remaining_s2.withColumn(
    "amount_bucket",
    (F.floor(F.col(FIN_AMOUNT_COL) / BUCKET_SIZE) * BUCKET_SIZE).cast("long")
)

# Expand axis buckets to ±1 for neighbour search
# This creates 3 rows per axis trade — much cheaper than cross-join
bucket_offsets = spark.createDataFrame(
    [(-BUCKET_SIZE,), (0,), (BUCKET_SIZE,)],
    ["_offset"]
)

axis_expanded = (
    axis_remaining_s2
    .crossJoin(F.broadcast(bucket_offsets))
    .withColumn("search_bucket", F.col("amount_bucket") + F.col("_offset"))
    .drop("_offset")
)

# Join on bucket
greedy2_candidates = (
    axis_expanded.alias("a")
    .join(
        fin_remaining_s2.alias("f"),
        on=(F.col("a.search_bucket") == F.col("f.amount_bucket")),
        how="inner"
    )
    # Compute diff and tolerance
    .withColumn("amount_diff", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}") - F.col(f"f.{FIN_AMOUNT_COL}")))
    .withColumn("tolerance", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")) * STRICT_TOLERANCE_PCT)
    # Filter strict tolerance
    .filter(F.col("amount_diff") <= F.col("tolerance"))
    # Deduplicate expanded rows (same axis_id+fin_id may come from multiple bucket offsets)
    .dropDuplicates(["axis_id", "fin_id"])
    .select(
        F.col("a.axis_id").alias("axis_id"),
        F.col("f.fin_id").alias("fin_id"),
        F.lit(17).cast("int").alias("priority"),
        F.lit("GREEDY").alias("category"),
        F.lit(2).cast("int").alias("brd_priority"),
        F.lit(f"Greedy: Amount Strict ({STRICT_TOLERANCE_PCT*100}%)").alias("description"),
        F.col("amount_diff"),
        (F.col("amount_diff") / F.greatest(F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")), F.lit(1e-9))).alias("amount_rel_diff"),
        F.lit(0).cast("int").alias("key_strength"),  # no key match — fuzzy
    )
)

# Resolve 1-to-1
greedy2_matches = resolve_one_to_one(greedy2_candidates)
greedy2_matches = greedy2_matches.withColumn("MatchLayer", F.lit("GREEDY"))
# Auditability columns (Best Practice §6)
greedy2_matches = (
    greedy2_matches
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
)
greedy2_matches = greedy2_matches.cache()

greedy2_count = greedy2_matches.count()
print(f"Strategy 2 matches: {greedy2_count:,}")

---
### 🔹 Section 14: Greedy Layer Summary

In [ ]:
# ============================================================
# LAYER 2 (GREEDY) SUMMARY
# ============================================================

total_greedy = greedy1_count + greedy2_count

print(f"\n{'='*60}")
print("LAYER 2 (GREEDY) SUMMARY")
print(f"{'='*60}")
print(f"Strategy 1 (Amount+Counterparty): {greedy1_count:,}")
print(f"Strategy 2 (Amount Strict):       {greedy2_count:,}")
print(f"Total Greedy matches:             {total_greedy:,}")

---
## 🔹 Section 15: Final Results Consolidation

In [ ]:
# ============================================================
# FINAL CONSOLIDATION
# ============================================================

# Union all match layers
all_matches = brd_matches.unionByName(greedy1_matches).unionByName(greedy2_matches)
all_matches = all_matches.cache()

total_matched = all_matches.count()
final_match_rate = total_matched / ORIGINAL_AXIS_COUNT * 100

# Final unmatched
all_matched_axis_ids = all_matches.select("axis_id")
all_matched_fin_ids = all_matches.select("fin_id")

final_unmatched_axis = axis_core.join(all_matched_axis_ids, on="axis_id", how="left_anti")
final_unmatched_fin = fin_core.join(all_matched_fin_ids, on="fin_id", how="left_anti")

unmatched_axis_count = final_unmatched_axis.count()
unmatched_fin_count = final_unmatched_fin.count()

print(f"\n{'='*80}")
print("FINAL RESULTS")
print(f"{'='*80}")
print(f"SCOPE: {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}")
print(f"{'-'*60}")
print(f"Total Axis (in-scope): {ORIGINAL_AXIS_COUNT:,}")
print(f"\nLayer 1 (BRD Rules):")
print(f"  {brd_match_count:,} matched ({brd_match_count/ORIGINAL_AXIS_COUNT*100:.2f}%)")
print(f"Layer 2 (Greedy):")
print(f"  {total_greedy:,} matched ({total_greedy/ORIGINAL_AXIS_COUNT*100:.2f}%)")
print(f"\nTOTAL MATCHED:           {total_matched:,}")
print(f"TOTAL UNMATCHED (Axis):  {unmatched_axis_count:,}")
print(f"TOTAL UNMATCHED (Fin):   {unmatched_fin_count:,}")
print(f"\n>>> FINAL MATCH RATE: {final_match_rate:.2f}% <<<")
print(f"{'='*80}")

---
## 🔹 Section 16: Enrichment — Join Back Wide Columns

Now that matching is done on narrow core tables,
we join back the full 100+ columns for the final output.

In [ ]:
# ============================================================
# ENRICHMENT — Join back wide columns (100+ cols)
# ============================================================
# This is the ONLY place where wide DataFrames are joined.
# All matching was done on narrow core tables.

# Rename wide columns to avoid collisions
axis_wide_renamed = axis_wide
for c in axis_wide.columns:
    if c != "axis_id":
        axis_wide_renamed = axis_wide_renamed.withColumnRenamed(c, f"{c}_Axis")

fin_wide_renamed = fin_wide
for c in fin_wide.columns:
    if c != "fin_id":
        fin_wide_renamed = fin_wide_renamed.withColumnRenamed(c, f"{c}_Finstore")

# Enrich matches
matches_enriched = (
    all_matches
    .join(axis_wide_renamed, on="axis_id", how="left")
    .join(fin_wide_renamed, on="fin_id", how="left")
)

# Add Variance column
axis_amt_col_enriched = f"{AXIS_AMOUNT_COL}_Axis"
fin_amt_col_enriched = f"{FIN_AMOUNT_COL}_Finstore"

matches_enriched = matches_enriched.withColumn(
    "Variance",
    F.col(axis_amt_col_enriched) - F.col(fin_amt_col_enriched)
)

# Enrich unmatched axis
unmatched_axis_enriched = (
    final_unmatched_axis
    .select("axis_id")
    .join(axis_wide_renamed, on="axis_id", how="left")
)

# Enrich unmatched finstore
unmatched_fin_enriched = (
    final_unmatched_fin
    .select("fin_id")
    .join(fin_wide_renamed, on="fin_id", how="left")
)

print(f"Enriched matches: {matches_enriched.count():,} rows, {len(matches_enriched.columns)} columns")
print(f"Enriched unmatched axis: {unmatched_axis_enriched.count():,} rows")
print(f"Enriched unmatched finstore: {unmatched_fin_enriched.count():,} rows")

---
## 🔹 Section 17: Matches by System Breakdown

In [ ]:
# ============================================================
# BREAKDOWN: Matches by Source System
# ============================================================

print(f"\n{'='*60}")
print("MATCHES BY SOURCE SYSTEM")
print(f"{'='*60}")

# BRD matches by system
print("\nBRD Matches by system:")
brd_by_system = (
    brd_matches
    .join(axis_core.select("axis_id", "SourceSystemName"), on="axis_id", how="left")
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
brd_by_system.show(20, truncate=False)

# Greedy matches by system
print("Greedy Matches by system:")
greedy_all = greedy1_matches.unionByName(greedy2_matches)
greedy_by_system = (
    greedy_all
    .join(axis_core.select("axis_id", "SourceSystemName"), on="axis_id", how="left")
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
greedy_by_system.show(20, truncate=False)

---
## 🔹 Section 18: Remaining Unmatched by System

In [ ]:
# ============================================================
# UNMATCHED BY SYSTEM
# ============================================================

print(f"\n{'='*60}")
print("REMAINING UNMATCHED BY SYSTEM")
print(f"{'='*60}")

unmatched_by_system = (
    final_unmatched_axis
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
unmatched_by_system.show(20, truncate=False)

---
## 🔹 Section 18b: Data Quality Validation (Best Practice §7)

Validate core column data contracts before outputs are consumed downstream.

In [ ]:
# ============================================================
# DATA QUALITY VALIDATION  (Best Practice §7)
# Validate core column contracts on final outputs.
# ============================================================

from functools import reduce

def validate_dataframe(df, name, checks):
    """Run a list of (description, filter_expr) checks.
    Returns a summary DF of violations found.
    """
    rows = []
    for desc, expr in checks:
        cnt = df.filter(expr).count()
        rows.append((name, desc, cnt))
    return spark.createDataFrame(rows, ["dataset", "check", "violation_count"])

# --- Axis Core checks ---
axis_checks = [
    ("null axis_id",            F.col("axis_id").isNull()),
    ("null CounterpartyId",     F.col("CounterpartyId").isNull()),
    ("null NotionalAmount",     F.col(AXIS_AMOUNT_COL).isNull()),
    ("negative Notional",       F.col(AXIS_AMOUNT_COL) < 0),
    ("null ReconSubProduct",    F.col("ReconSubProduct").isNull()),
]

# --- Finstore Core checks ---
fin_checks = [
    ("null fin_id",             F.col("fin_id").isNull()),
    ("null counterpartyid",     F.col("counterpartyid").isNull()),
    ("null NotionalAmount",     F.col(FIN_AMOUNT_COL).isNull()),
    ("negative Notional",       F.col(FIN_AMOUNT_COL) < 0),
]

dq_axis = validate_dataframe(axis_core, "axis_core", axis_checks)
dq_fin  = validate_dataframe(fin_core,  "fin_core",  fin_checks)

# --- Match output checks ---
match_checks = [
    ("null axis_id",        F.col("axis_id").isNull()),
    ("null fin_id",         F.col("fin_id").isNull()),
    ("null description",    F.col("description").isNull()),
    ("null run_id",         F.col("run_id").isNull()),
    ("amount_rel_diff > 1", F.col("amount_rel_diff") > 1.0),
]
dq_matches = validate_dataframe(all_matches, "all_matches", match_checks)

dq_report = dq_axis.union(dq_fin).union(dq_matches)
print("=== Data Quality Validation ===")
dq_report.show(50, truncate=False)

violations = dq_report.filter(F.col("violation_count") > 0).count()
if violations > 0:
    print(f"⚠️  {violations} check(s) have violations — review before promoting to Gold.")
else:
    print("✅ All data quality checks passed.")

---
## 🔹 Section 18c: Explainability — Unmatched Reason Breakdown (Best Practice §6)

Every unmatched trade should carry a *reason* so downstream consumers can investigate.

In [ ]:
# ============================================================
# EXPLAINABILITY — UNMATCHED REASON BREAKDOWN  (Best Practice §6)
# Classify why each unmatched trade failed to match.
# ============================================================

# --- Axis unmatched reasons ---
# Check if the trade appeared as a candidate in ANY rule but was outcompeted
axis_candidates_ever = candidates_layer1.select("axis_id").distinct()

axis_unmatched_reasons = (
    final_unmatched_axis
    .join(axis_candidates_ever, on="axis_id", how="left")
    .withColumn(
        "unmatched_reason",
        F.when(axis_candidates_ever["axis_id"].isNotNull(),
               F.lit("candidate_existed_but_consumed_by_higher_priority"))
         .otherwise(F.lit("no_candidate_key_found"))
    )
    .select("axis_id", "SourceSystemTradeId", "CounterpartyId", "ReconSubProduct",
            AXIS_AMOUNT_COL, "unmatched_reason")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
)

# --- Finstore unmatched reasons ---
fin_candidates_ever = candidates_layer1.select("fin_id").distinct()

fin_unmatched_reasons = (
    final_unmatched_fin
    .join(fin_candidates_ever, on="fin_id", how="left")
    .withColumn(
        "unmatched_reason",
        F.when(fin_candidates_ever["fin_id"].isNotNull(),
               F.lit("candidate_existed_but_consumed_by_higher_priority"))
         .otherwise(F.lit("no_candidate_key_found"))
    )
    .select("fin_id", "tradeid", "counterpartyid",
            FIN_AMOUNT_COL, "unmatched_reason")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
)

# --- Summary ---
print("=== Unmatched Axis Reason Breakdown ===")
axis_unmatched_reasons.groupBy("unmatched_reason").count().show(truncate=False)

print("=== Unmatched Finstore Reason Breakdown ===")
fin_unmatched_reasons.groupBy("unmatched_reason").count().show(truncate=False)

# Optionally persist for audit
# axis_unmatched_reasons.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_reasons")
# fin_unmatched_reasons.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_fin_reasons")

---
## 🔹 Section 19: Save Results

**Best practice:** Save as Delta for downstream consumption;
export CSV only for legacy consumers / samples.

In [ ]:
# ============================================================
# SAVE RESULTS — Delta (primary) + CSV (optional export)
# ============================================================

print(f"Saving results to: {OUTPUT_DIR}")
print(f"{'-'*60}")

# ---- Delta outputs (recommended for production) ----

# BRD matches (narrow — match metadata)
brd_matches.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_brd_layer")
print(f"Saved Delta: matched_brd_layer ({brd_match_count:,} rows)")

# Greedy matches (narrow)
greedy_all_df = greedy1_matches.unionByName(greedy2_matches)
greedy_all_count = greedy_all_df.count()
greedy_all_df.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_greedy_layer")
print(f"Saved Delta: matched_greedy_layer ({greedy_all_count:,} rows)")

# All matches enriched (wide — full 100+ columns)
matches_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_all_combined")
print(f"Saved Delta: matched_all_combined ({total_matched:,} rows)")

# Unmatched Axis (wide)
unmatched_axis_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_final")
print(f"Saved Delta: unmatched_axis_final ({unmatched_axis_count:,} rows)")

# Unmatched Finstore (wide)
unmatched_fin_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_finstore_final")
print(f"Saved Delta: unmatched_finstore_final ({unmatched_fin_count:,} rows)")

# ---- OPTIMIZE + ZORDER (Best Practice §5 — Partitioning & Layout) ----
# Run these after writing to optimise read performance for downstream queries.
# Uncomment on Databricks (requires Delta Lake OPTIMIZE support):
#
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/matched_all_combined` ZORDER BY (axis_id, fin_id, priority)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/unmatched_axis_final` ZORDER BY (axis_id)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/unmatched_finstore_final` ZORDER BY (fin_id)")
# print("Delta tables OPTIMIZED + ZORDERed.")

# ---- Optional CSV exports (e.g., for stakeholder review / Excel) ----
# Uncomment below to also save CSV (coalesce(1) for single file):

# matches_enriched.coalesce(1).write.mode("overwrite").option("header", "true") \
#     .csv(f"{OUTPUT_DIR}/csv/matched_all_combined")
# unmatched_axis_enriched.coalesce(1).write.mode("overwrite").option("header", "true") \
#     .csv(f"{OUTPUT_DIR}/csv/unmatched_axis_final")

print("\nAll results saved successfully.")

---
## 🔹 Section 20: Summary Report

In [ ]:
# ============================================================
# SUMMARY REPORT
# ============================================================

report = f"""
{'='*80}
DERIVATIVES MATCHING — COMPLETE RESULTS (BRD + GREEDY)
PySpark / Databricks Production
{'='*80}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Run ID:    {RUN_ID}
Batch ID:  {BATCH_ID}
Rule Ver:  {RULE_VERSION}
Scope:     {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}

{'='*80}
INPUT DATA
{'='*80}
Total Axis (original):  {ORIGINAL_AXIS_COUNT_FULL:,}
Total Finstore:         {ORIGINAL_FINSTORE_COUNT:,}
Excluded (SOPHIS/DELTA1): {excluded_count:,}
Axis (in-scope):        {ORIGINAL_AXIS_COUNT:,}

{'='*80}
LAYER 1: BRD DETERMINISTIC MATCHING
{'='*80}
Rules executed: 15 (3 SOPHIS + 10 OTC + 2 ETD)
Unique Axis matched: {brd_match_count:,}
Match Rate: {brd_match_rate:.2f}%

{'='*80}
LAYER 2: GREEDY/PROBABILISTIC MATCHING
{'='*80}
Strategy 1 (Amount+Counterparty, {GREEDY_TOLERANCE_PCT*100:.1f}%): {greedy1_count:,}
Strategy 2 (Amount Strict, {STRICT_TOLERANCE_PCT*100:.1f}%): {greedy2_count:,}
Total Greedy: {total_greedy:,}

{'='*80}
COMBINED RESULTS
{'='*80}
Total Matched:   {total_matched:,}
Total Unmatched: {unmatched_axis_count:,}
Final Match Rate: {final_match_rate:.2f}%

{'='*80}
SCORING & AUDITABILITY (Best Practice §2D, §6)
{'='*80}
- Every match row includes: amount_diff, amount_rel_diff, key_strength
- Auditability columns: run_id, batch_id, rule_version, match_timestamp
- Unmatched reason breakdown: see Explainability section above

{'='*80}
PERFORMANCE OPTIMISATIONS APPLIED
{'='*80}
- Core/Wide split: matching on {len(AXIS_CORE_COLS)} Axis + {len(FIN_CORE_COLS)} Finstore columns
- Candidate edges + 3-pass window ranking (no iterative pool removal)
- Blocked joins for greedy (counterparty / amount buckets)
- Native Spark SQL functions (no Python UDFs)
- AQE + skew join handling enabled
- Delta output format with ZORDER optimisation

{'='*80}
DATA QUALITY (Best Practice §7)
{'='*80}
- Core column null/range checks executed (see DQ Validation section above)
{'='*80}
"""

print(report)

# Save report as text
report_rdd = spark.sparkContext.parallelize([report])
report_rdd.coalesce(1).saveAsTextFile(f"{OUTPUT_DIR}/matching_summary_report")
print(f"Summary report saved to: {OUTPUT_DIR}/matching_summary_report")

---
## 🔹 Section 21: Cleanup — Unpersist Cached DataFrames

In [ ]:
# ============================================================
# CLEANUP — release cached DataFrames
# ============================================================

for df_to_unpersist in [axis_core, fin_core, candidates_layer1,
                         brd_matches, axis_unmatched, fin_unmatched,
                         greedy1_matches, greedy2_matches, all_matches]:
    try:
        df_to_unpersist.unpersist()
    except Exception:
        pass

print("Cached DataFrames unpersisted. Pipeline complete.")